# The Script: WR Player-Game Data Exploration

> Given information available before kickoff, The Script predicts a wide receiver's receiving yards in an upcoming NFL game.



**Prediction time:** 24 hours before kickoff  

**Target:** `receiving_yards`  

**Population:** recorded regular-season WR-games, 2016-2025



This notebook audits the canonical base table only. It does not engineer rolling features or train a model.

In [1]:
from pathlib import Path



import polars as pl



PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

TABLE_PATH = PROJECT_ROOT / "data" / "processed" / "wr_player_games.parquet"

player_games = pl.read_parquet(TABLE_PATH)

player_games.shape

(23610, 14)

## Inspect Ten Rows

Confirm player IDs, teams, schedule-derived opponents, dates, home/away status, rest, and receiving outcomes before feature engineering.

In [2]:
player_games.head(10)

game_id,game_date,season,week,player_id,team,opponent,home_game,days_rest,position,receiving_yards,targets,receptions,receiving_touchdowns
str,date,i32,i32,str,str,str,bool,i32,str,i32,i32,i32,i32
"""2016_01_CAR_DEN""",2016-09-08,2016,1,"""00-0025396""","""CAR""","""DEN""",false,7,"""WR""",5,1,1,0
"""2016_01_CAR_DEN""",2016-09-08,2016,1,"""00-0026739""","""DEN""","""CAR""",true,7,"""WR""",0,1,0,0
"""2016_01_CAR_DEN""",2016-09-08,2016,1,"""00-0027685""","""DEN""","""CAR""",true,7,"""WR""",49,8,5,0
"""2016_01_CAR_DEN""",2016-09-08,2016,1,"""00-0027874""","""DEN""","""CAR""",true,7,"""WR""",48,6,4,0
"""2016_01_CAR_DEN""",2016-09-08,2016,1,"""00-0031118""","""CAR""","""DEN""",false,7,"""WR""",11,2,1,0
"""2016_01_CAR_DEN""",2016-09-08,2016,1,"""00-0031344""","""DEN""","""CAR""",true,7,"""WR""",0,0,0,0
"""2016_01_CAR_DEN""",2016-09-08,2016,1,"""00-0031359""","""CAR""","""DEN""",false,7,"""WR""",91,12,6,1
"""2016_01_CAR_DEN""",2016-09-08,2016,1,"""00-0032055""","""CAR""","""DEN""",false,7,"""WR""",9,4,1,0
"""2016_01_BUF_BAL""",2016-09-11,2016,1,"""00-0020337""","""BAL""","""BUF""",true,7,"""WR""",19,8,5,0


In [3]:
canonical_ids = ["game_id", "game_date", "player_id", "team", "opponent"]



quality_summary = player_games.select(

    pl.len().alias("rows"),

    pl.struct(["game_id", "player_id"]).n_unique().alias("unique_player_games"),

    pl.sum_horizontal(pl.col(canonical_ids).null_count()).alias("null_canonical_ids"),

    pl.col("season").min().alias("first_season"),

    pl.col("season").max().alias("last_season"),

    pl.col("receiving_yards").min().alias("min_receiving_yards"),

    pl.col("receiving_yards").max().alias("max_receiving_yards"),

    pl.col("home_game").mean().alias("home_share"),

)



assert quality_summary["rows"][0] == quality_summary["unique_player_games"][0]

assert quality_summary["null_canonical_ids"][0] == 0

assert quality_summary["first_season"][0] == 2016

assert quality_summary["last_season"][0] == 2025

quality_summary

rows,unique_player_games,null_canonical_ids,first_season,last_season,min_receiving_yards,max_receiving_yards,home_share
u32,u32,u32,i32,i32,i32,i32,f64
23610,23610,0,2016,2025,-13,300,0.499322


In [4]:
player_games.group_by("season").len().sort("season")

season,len
i32,u32
2016,2286
2017,2239
2018,2226
2019,2256
2020,2305
2021,2483
2022,2387
2023,2476
2024,2441


## Audit Notes

- The table contains 23,610 unique WR-game rows across every season from 2016 through 2025.

- Weekly player statistics are the current participation proxy. Snap-count confirmation is the next source join.

- Negative receiving yards are legitimate NFL outcomes and must not be rejected as invalid targets.

- No predictive features exist in this table yet, so current-game receiving outcomes cannot leak into feature columns.

- Feature engineering and model training remain blocked until this table is accepted.